# Lesson 02｜網路原理、爬蟲倫理與環境

今天先搞懂「瀏覽器怎麼拿到網頁」，再用 Python 做同樣的事。
在 VS Code 按每格上方的 Run Cell 逐格執行。沒有網路時，直接跳到最後一格。


## 本章學習地圖

**準備工具與複習語法** → **認識網頁與請求** → **抓取、檢查、存檔**

| 階段 | 會用到的語法或工具 | 完成後能做什麼 |
|---|---|---|
| 課前複習 | 字串、變數、dict `.get()`、`if / else`、函式與 `try / except` | 能閱讀網址、狀態碼與程式執行結果 |
| 環境與網路 | Python、VS Code 的 Python 與 Jupyter 擴充功能、`.venv`、Notebook kernel、瀏覽器 F12 | 用 VS Code 選取課程環境，認識 URL、HTTP 請求、回應與 HTML |
| 第一次抓網頁 | `requests.get(url, timeout=10)`、`status_code`、`headers`、`text / content` | 抓取一頁、檢查回應，並觀察 HTML |
| 保存與負責任爬取 | `Path`、檔案讀寫、`raise_for_status()`、robots.txt、User-Agent | 將網頁存檔讀回，認識錯誤處理與爬蟲禮儀 |

**課堂產出：**執行 Notebook，取得一個練習網頁、檢查狀態碼並存成 HTML。若無法連網，可使用課程提供的本機快照。


### Notebook 使用方式

先選擇課程 Python 環境，再由上到下執行每一格（Shift + Enter）。先執行工作資料夾設定及前面的匯入。
若看到 `NameError`，先確認變數名稱，並回到上方執行建立該變數的格子；必要時重新啟動核心並從頭執行。練習或作業格要先完成 TODO，再執行檢查。


## 設定工作資料夾

請先執行下一格。使用 VS Code 開啟完整教材資料夾；本堂的 `fixtures/` 和 `output/` 會以本堂資料夾為基準。


In [ ]:
import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson02_network_and_environment'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())


## 01｜今天的成果預覽

先看結果，再拆解程式：用 Python 抓回一個網頁。


In [ ]:
import requests

url = "https://quotes.toscrape.com/"
response = requests.get(url, timeout=10)
print("狀態碼：", response.status_code)
print("網頁長度：", len(response.text), "個字元")


## 02｜瀏覽器打開網頁的五個步驟

爬蟲做的事，就是自己完成第 3、4 步，不需要第 5 步的畫面。


In [ ]:
steps = [
    "輸入網址 URL",
    "用主機名稱找到伺服器的位置（DNS）",
    "送出 HTTP 請求 request",
    "收到 HTTP 回應 response（內容是 HTML）",
    "瀏覽器把 HTML 畫成畫面",
]
for number, step in enumerate(steps, start=1):
    print(number, step)


## 03｜URL 的組成

同一個網站的不同頁面，差別通常就在 path 與 query。


In [ ]:
from urllib.parse import parse_qs, urlparse

example_url = "https://quotes.toscrape.com/tag/love/page/2/?sort=new"
parts = urlparse(example_url)
print("通訊協定 scheme：", parts.scheme)
print("主機 host：", parts.netloc)
print("路徑 path：", parts.path)
print("查詢 query：", parts.query)
print("query 轉成字典：", parse_qs(parts.query))


## 04｜常見的 HTTP 狀態碼

爬蟲每次請求後，第一件事就是看狀態碼。


In [ ]:
status_meaning = {
    200: "成功",
    301: "網址搬家了（重新導向）",
    403: "被拒絕：沒有權限，或被網站擋下",
    404: "找不到頁面",
    429: "請求太頻繁",
    500: "網站伺服器出錯",
}
for code in [200, 404, 403]:
    print(code, status_meaning.get(code, "不認識的狀態碼"))


## 05｜用 F12 看瀏覽器做了什麼（課堂操作，無程式）

1. 用 Chrome 打開 https://quotes.toscrape.com/ ，按 F12。
2. 切到 **Network**，按 Ctrl+R（Mac 是 Cmd+R）重新整理。
3. 點第一筆 request，看 Headers：Request URL、Request Method、Status Code。
4. 切到 **Elements**，把滑鼠移到網頁上的一則名言，看對應的 HTML。


### 想多練幾個網站？

先用瀏覽器和 F12 觀察，再用 `requests.get(url, timeout=10)` 取回一頁：

| 網站 | 練習方向 | 難度 |
|---|---|---|
| https://quotes.toscrape.com/ | 名言與分頁；課堂主要範例 | 入門 |
| https://books.toscrape.com/ | 書籍目錄與分頁 | 入門 |
| https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html | 單筆商品頁與網址路徑 | 入門 |
| https://httpbin.org/html | 簡單 HTML、狀態碼、Content-Type | 入門 |
| https://jsonplaceholder.typicode.com/posts | JSON API；比較 `.text` 和 `.json()` | 入門＋ |

Lesson 02 先練取得回應；解析欄位留到後續課程。遵守網站條款與 `robots.txt`，
每次只請求一頁並設定 timeout。遇到 403／429 就停，不要嘗試繞過限制。


## 06｜確認 Notebook 的環境

如果已用 VS Code 的介面建立環境、安裝套件，並在右上角選擇課程的 `.venv` 核心，執行下一格確認即可。

有印出 Python 路徑與 requests 版本，就可以繼續。如果顯示找不到 requests，請先確認右上角的核心，再檢查是否已將套件安裝到該環境。

### 選修：用終端機建立環境

如果圖形介面設定遇到問題，可以參考 Lesson 02 投影片中的 Mac 或 Windows 終端機備用方式。請在 VS Code 終端機輸入，不是在 Notebook 程式格執行。


In [ ]:
import sys

print("目前使用的 Python：", sys.executable)
try:
    import requests
    print("requests 版本：", requests.__version__)
except ModuleNotFoundError:
    print("找不到 requests：請確認右上角選的是課程 .venv 核心，並將 requests 安裝到該環境。")


## 07｜Response 物件的重點欄位


In [ ]:
print(response)
print("status_code：", response.status_code)
print("ok：", response.ok)
print("Content-Type：", response.headers["Content-Type"])
print("encoding：", response.encoding)
print("最終網址：", response.url)


## 08｜看看 HTML 長什麼樣子


In [ ]:
html = response.text
print(html[:300])


## 09｜先用字串工具找內容

能找，但很不方便。這就是第 4 堂要學 BeautifulSoup 的原因。


In [ ]:
print("有沒有 <title>：", "<title>" in html)
print("頁面上的名言數量：", html.count('class="quote"'))

start = html.find('<span class="text"')
print(html[start : start + 150])


## 10｜把網頁存成檔案

`response.text` 是已解碼的文字，`response.content` 是原始 bytes。
要原樣保存整個網頁，存 `content`。


In [ ]:
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
save_path = output_dir / "quotes_page1.html"
save_path.write_bytes(response.content)
print("已存檔：", save_path, "，大小：", save_path.stat().st_size, "bytes")


## 11｜讀回本機檔案


In [ ]:
saved_html = save_path.read_text(encoding="utf-8")
print("讀回的名言數量：", saved_html.count('class="quote"'))


## 12｜找不到頁面：404 與 raise_for_status


In [ ]:
bad_response = requests.get("https://quotes.toscrape.com/no-such-page", timeout=10)
print("狀態碼：", bad_response.status_code, "，ok：", bad_response.ok)

try:
    bad_response.raise_for_status()
except requests.HTTPError as error:
    print("捕捉到 HTTPError：", error)


## 13｜robots.txt：網站寫給爬蟲看的規則

這裡用一段自己寫的內容示範，結果每次都一樣。


In [ ]:
from urllib.robotparser import RobotFileParser

robots_text = """User-agent: *
Disallow: /admin/
Disallow: /search
Allow: /
"""
parser = RobotFileParser()
parser.parse(robots_text.splitlines())

for path in ["/", "/admin/users", "/search?q=python"]:
    print(path, "可以抓嗎？", parser.can_fetch("*", path))


## 14｜表明身分：User-Agent


In [ ]:
print("預設送出的 User-Agent：", response.request.headers["User-Agent"])

my_headers = {"User-Agent": "python-scraping-viz-class/0.1 (student project)"}
polite_response = requests.get(url, headers=my_headers, timeout=10)
print("自訂後送出的 User-Agent：", polite_response.request.headers["User-Agent"])


## 15｜爬取守則


In [ ]:
rules = [
    "先看網站有沒有開放資料或 API，再考慮爬網頁",
    "看 robots.txt 與服務條款，不確定就不要抓",
    "設定 timeout，連續請求之間加 time.sleep",
    "只抓分析需要的欄位，不公開轉載別人的內文",
    "被擋（403、429）就停下來，不要想辦法繞過",
]
for number, rule in enumerate(rules, start=1):
    print(number, rule)


## 16｜小結：抓取、檢查、存檔


In [ ]:
import time

book_url = "https://books.toscrape.com/"
book_response = requests.get(book_url, headers=my_headers, timeout=10)

if book_response.ok:
    book_path = output_dir / "books_index.html"
    book_path.write_bytes(book_response.content)
    print("已存檔：", book_path, "，共", len(book_response.content), "bytes")
else:
    print("失敗，狀態碼：", book_response.status_code)

time.sleep(1)


## 17｜沒網路也能上課

`fixtures/` 裡有事先存好的網頁，讀它就能繼續練習。


In [ ]:
fixture_html = Path("fixtures/quotes_page1.html").read_text(encoding="utf-8")
print("本機快照的名言數量：", fixture_html.count('class="quote"'))
